# 06 EDA: Injury Reported target analysis

End-to-end binary-classification EDA against `Injury Reported` (~9.5% positive rate, 222 / 2344 rows). Drives three new utils:

1. `eda_utils_target_univariate` -- basic EDA + univariate ranking (AUC, KS, MI, chi2, correlation)
2. `eda_utils_target_models` -- LightGBM RF + Logistic Regression + LightGBM GBM + SHAP
3. `eda_utils_target_interactions` -- pairwise target-rate heatmaps + stub depth-3 decision tree

All artifacts land under `artifacts_target_injury/<section>/` so results survive a kernel restart.

**Env**: activate the Python 3.12 sidecar before launching jupyter:
```bash
source ~/claude_code_repos/my-uv-envs/avird-2026-eda-target/.venv/Scripts/activate
```
LightGBM + SHAP do not ship Python 3.14 wheels here, hence the sidecar. The main `avird-2026-eda` env will not have these packages.

## Section 0 -- Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append('..')

import eda_utils_sgo
import eda_utils_dedupe
import eda_utils_treatment
import eda_utils_targets as tgt
import eda_utils_target_univariate as ut
import eda_utils_target_models as mdl
import eda_utils_target_interactions as ix

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 300)

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# Where artifacts land. Re-runs overwrite -- the notebook is the source of truth.
ARTIFACT_ROOT = Path('artifacts_target_injury')
ARTIFACT_ROOT.mkdir(exist_ok=True)

TARGET_COL = 'Injury Reported'
RANDOM_STATE = 0
TOP_K_INTERACTIONS = 8

## Section 1 -- Load + treat + attach targets

Copies the preamble from `04_eda_target_exploration.ipynb` verbatim, then selects `Injury Reported` as the binary target for the rest of the notebook.

In [4]:
repo_root = Path.cwd().parents[1]  # eda/ADS_to_2026_03_16 -> repo root
data_dir = repo_root / 'data' / 'nhtsa'
paths = [
    data_dir / 'SGO-2021-01_Incident_Reports_ADS_to_2025_06_16.csv',
    data_dir / 'SGO-2021-01_Incident_Reports_ADS_2025_06_16_to_2026_03_16.csv',
]
ads_df = eda_utils_sgo.load_and_concat_csvs(paths)

Only in SGO-2021-01_Incident_Reports_ADS_to_2025_06_16.csv:
  ADAS/ADS Hardware Version
  ADAS/ADS Hardware Version - Unk
  ADAS/ADS Hardware Version CBI
  ADAS/ADS Software Version
  ADAS/ADS Software Version - Unk
  ADAS/ADS Software Version CBI
  ADAS/ADS System Version
  ADAS/ADS System Version - Unk
  ADAS/ADS System Version CBI
  ADS Equipped?
  CP Any Air Bags Deployed?
  CP Was Vehicle Towed?
  Federal Reg. Exemption - No
  Federal Reg. Exemption - Unk
  Federal Regulatory Exemption?
  Inv. Officer Email - Unknown
  Inv. Officer Name - Unknown
  Inv. Officer Phone - Unknown
  Investigating Officer Email
  Investigating Officer Name
  Investigating Officer Phone
  Law Enforcement Investigating?
  Lighting
  Mileage
  Mileage - Unknown
  Notice Received Date
  Other Federal Reg. Exemption
  Other Reporting Entities?
  Other Reporting Entities? - NA
  Other Reporting Entities? - Unk
  Posted Speed Limit (MPH)
  Posted Speed Limit - Unknown
  Property Damage?
  Rep Ent Or Mfr Inves

In [5]:
treated_df = eda_utils_dedupe.dedupe_same_incident(ads_df.copy(), verbose=True)
treated_df = eda_utils_treatment.apply_all_treatments(treated_df)
treated_df = tgt.add_all_targets(treated_df, sv_speed_threshold=15)

print(f'rows: {len(treated_df)}')
print(f'{TARGET_COL!r} positive rate: {treated_df[TARGET_COL].mean():.4f}')
print(f'{TARGET_COL!r} positive count: {int(treated_df[TARGET_COL].sum())}')

dedupe_same_incident: 3120 -> 2344 rows (776 duplicates collapsed)
rows: 2344
'Injury Reported' positive rate: 0.0947
'Injury Reported' positive count: 222


## Section 2 -- Feature schema

`default_feature_columns` drops the static drop-list (free-text / IDs / officer PII) plus the target's source columns (`Highest Injury Severity Alleged` for `Injury Reported`) plus the other six generated target columns. The remaining set goes into every downstream model + univariate scorer.

In [6]:
feature_cols = ut.default_feature_columns(treated_df, TARGET_COL)
print(f'feature_cols: {len(feature_cols)} columns')
print(f'head: {feature_cols[:10]}')
print(f'tail: {feature_cols[-10:]}')

# Persist the list for downstream review (which cols entered the model?)
(ARTIFACT_ROOT / 'feature_cols.txt').write_text(
    '\n'.join(feature_cols) + '\n', encoding='utf-8'
)

feature_cols: 153 columns
head: ['Reporting Entity', 'Report Type', 'Report Month', 'Report Year', 'Report Submission Date', 'VIN - Unknown', 'Make', 'Model', 'Model - Unknown', 'Model Year']
tail: ['Were All Passengers Belted?', 'Operating Entity Clean', 'Reporting Entity Clean', 'Investigating Agency Clean', 'State or Local Permit Clean', 'Make Clean', 'Model Clean', 'State Clean', 'master_entity', 'Make Model']


3477

In [7]:
# Co-observed crash-outcome features: NOT pre-incident, so they may correlate
# with injury via co-measurement rather than causation. Section 6 runs a
# 'pre-incident only' contrast SHAP pass against this reduced feature set.
CO_OBSERVED_OUTCOME_COLS = (
    'Was Any Vehicle Towed?',
    'CP Was Vehicle Towed?',
    'SV Was Vehicle Towed?',
    'Any Air Bags Deployed?',
    'CP Any Air Bags Deployed?',
    'SV Any Air Bags Deployed?',
    'SV Precrash Speed (MPH)',
)
pre_incident_feature_cols = [c for c in feature_cols if c not in CO_OBSERVED_OUTCOME_COLS]
print(f'pre_incident_feature_cols: {len(pre_incident_feature_cols)} columns '
      f'({len(feature_cols) - len(pre_incident_feature_cols)} co-observed dropped)')

pre_incident_feature_cols: 146 columns (7 co-observed dropped)


## Section 3 -- Basic EDA against target

Per-feature value counts + describe, segmented by target value, rendered **inline** -- value counts in wide form (`count` and `share_within_target` for each target value side by side, one row per feature value) next to `describe`. No more clicking through hundreds of CSVs; pass `out_dir=` to `basic_eda_by_target` only when you want the artifacts on disk.

In [11]:
treated_df[TARGET_COL].mean()

np.float64(0.0947098976109215)

In [10]:
# Basic EDA inline: value counts (wide -- count + share per target side by
# side) and describe rendered together per feature, so we read results here
# instead of clicking through hundreds of CSVs. Returns a dict of the tables
# for reuse; pass out_dir=... to ALSO write artifacts (off by default).
basic_eda = ut.basic_eda_by_target(
    treated_df, TARGET_COL, feature_cols, top_n=20,
)
print(f'rendered basic EDA for {len(basic_eda)} features inline')

,pos_rate,share_within_target=0,share_within_target=1,count=0,count=1
feature_value,,,,,
Waymo LLC,0.084381,0.721018,0.635135,1530,141
"General Motors, LLC",0.163265,0.057964,0.108108,123,24
Transdev Alternative Services,0.060606,0.058435,0.036036,124,8
"Zoox, Inc.",0.240000,0.044769,0.135135,95,30
Avride Inc.,0.050000,0.017908,0.009009,38,2
May Mobility,0.026316,0.017436,0.004505,37,1
PACCAR Incorporated,0.000000,0.008954,0.000000,19,0
Ford Motor Company,0.176471,0.006598,0.013514,14,3
"Tesla, Inc.",0.133333,0.006126,0.009009,13,2


,pos_rate,share_within_target=0,share_within_target=1,count=0,count=1
feature_value,,,,,
NaN,0.166311,0.368520,0.702703,782,156
12.0,0.047945,0.065504,0.031532,139,7
2.0,0.031496,0.057964,0.018018,123,4
11.0,0.031496,0.057964,0.018018,123,4
8.0,0.055556,0.056079,0.031532,119,7
10.0,0.016393,0.056550,0.009009,120,2
1.0,0.041322,0.054665,0.022523,116,5
9.0,0.041322,0.054665,0.022523,116,5
5.0,0.035088,0.051838,0.018018,110,4


,pos_rate,share_within_target=0,share_within_target=1,count=0,count=1
feature_value,,,,,
JAN-2026,0.106061,0.055608,0.063063,118,14
MAY-2025,0.082569,0.047125,0.040541,100,9
DEC-2025,0.094340,0.045240,0.045045,96,10
JUN-2025,0.069307,0.044298,0.031532,94,7
NOV-2025,0.131313,0.040528,0.058559,86,13
FEB-2026,0.115789,0.039585,0.049550,84,11
OCT-2025,0.102273,0.037229,0.040541,79,9
JAN-2025,0.107143,0.035344,0.040541,75,9
APR-2025,0.037500,0.036287,0.013514,77,3


,pos_rate,share_within_target=0,share_within_target=1,count=0,count=1
feature_value,,,,,
Jaguar,0.068093,0.451461,0.315315,958,70
JAGUAR,0.105109,0.288878,0.324324,613,72
Cruise,0.181208,0.057493,0.121622,122,27
Toyota,0.206897,0.043355,0.108108,92,24
JLR,0.066667,0.026390,0.018018,56,4
HYUNDAI,0.045455,0.019793,0.009009,42,2
TOYOTA,0.233333,0.010839,0.031532,23,7
Peterbilt,0.043478,0.010368,0.004505,22,1
Hyundai,0.090909,0.009425,0.009009,20,2


,pos_rate,share_within_target=0,share_within_target=1,count=0,count=1
feature_value,,,,,
I-PACE,0.099482,0.409519,0.432432,869,96
I-Pace,0.062963,0.357681,0.229730,759,51
AV,0.182432,0.057022,0.121622,121,27
Highlander,0.287129,0.033930,0.130631,72,29
Ioniq 5,0.057692,0.023091,0.013514,49,3
Sienna,0.033333,0.013666,0.004505,29,1
579,0.000000,0.009896,0.000000,21,0
Zoox,0.000000,0.007540,0.000000,16,0
Model Y,0.133333,0.006126,0.009009,13,2


...

,pos_rate,share_within_target=0,share_within_target=1,count=0,count=1
feature_value,,,,,
i pace,0.082770,0.767672,0.662162,1629,147
av,0.182432,0.057022,0.121622,121,27
highlander,0.287129,0.033930,0.130631,72,29
ioniq 5,0.066667,0.026390,0.018018,56,4
sienna,0.033333,0.013666,0.004505,29,1
579,0.000000,0.009896,0.000000,21,0
zoox,0.000000,0.007540,0.000000,16,0
model y,0.133333,0.006126,0.009009,13,2
pacifica,0.000000,0.006598,0.000000,14,0


,pos_rate,share_within_target=0,share_within_target=1,count=0,count=1
feature_value,,,,,
CA,0.098522,0.603676,0.630631,1281,140
AZ,0.088496,0.194156,0.180180,412,40
TX,0.063670,0.117813,0.076577,250,17
NV,0.179104,0.025919,0.054054,55,12
GA,0.145833,0.019321,0.031532,41,7
FL,0.081081,0.016023,0.013514,34,3
MI,0.100000,0.004241,0.004505,9,1
DC,0.125000,0.003299,0.004505,7,1
MN,0.000000,0.002828,0.000000,6,0


,pos_rate,share_within_target=0,share_within_target=1,count=0,count=1
feature_value,,,,,
Waymo,0.082640,0.779453,0.671171,1654,149
Cruise,0.171975,0.061263,0.121622,130,27
Zoox,0.240000,0.044769,0.135135,95,30
Avride,0.050000,0.017908,0.009009,38,2
MAY Mobility,0.025641,0.017908,0.004505,38,1
Motional,0.090909,0.009425,0.009009,20,2
Aurora Operations,0.000000,0.009896,0.000000,21,0
Argo AI,0.150000,0.008011,0.013514,17,3
Tesla,0.133333,0.006126,0.009009,13,2


,pos_rate,share_within_target=0,share_within_target=1,count=0,count=1
feature_value,,,,,
jaguar | i pace,0.082896,0.740339,0.639640,1571,142
cruise | av,0.183673,0.056550,0.121622,120,27
toyota | highlander,0.287129,0.033930,0.130631,72,29
jlr | i pace,0.066667,0.026390,0.018018,56,4
hyundai | ioniq 5,0.067797,0.025919,0.018018,55,4
toyota | sienna,0.033333,0.013666,0.004505,29,1
peterbilt | 579,0.000000,0.009896,0.000000,21,0
zoox | zoox,0.000000,0.007540,0.000000,16,0
tesla | model y,0.133333,0.006126,0.009009,13,2


rendered basic EDA for 153 features inline


In [ ]:
# Reuse a single feature's already-computed tables from the returned dict:
display(basic_eda['Crash With']['value_counts'])
display(basic_eda['Crash With']['describe'])

# Artifacts are opt-in -- uncomment to persist CSVs for offline review:
# ut.basic_eda_by_target(treated_df, TARGET_COL, feature_cols,
#                        out_dir=ARTIFACT_ROOT / 'basic_eda', show=False)

## Section 4 -- Univariate ranking

AUC (magnitude + direction), KS, mutual information, chi-square p-value, Spearman correlation -- one row per feature. Sorted by mutual information desc. Numeric-only metrics carry NaN on categorical rows and vice versa.

In [10]:
ranking = ut.rank_features(treated_df, TARGET_COL, feature_cols)
(ARTIFACT_ROOT / 'univariate').mkdir(exist_ok=True)
ranking.to_csv(ARTIFACT_ROOT / 'univariate' / 'feature_ranking.csv', index=False)

print(f'ranked {len(ranking)} features')
display(ranking.head(30))

ranked 154 features


,feature,dtype,n_non_null,auc,auc_direction,ks,mutual_info,chi2_p,correlation
0,Incident Time (24:00),str,2339,NaN,NaN,NaN,0.201118,2.951014e-03,NaN
1,Investigating Agency,str,509,NaN,NaN,NaN,0.054752,5.414517e-34,NaN
2,Investigating Agency Clean,str,508,NaN,NaN,NaN,0.054117,9.287501e-34,NaN
3,ADAS/ADS System Version,str,1514,NaN,NaN,NaN,0.036651,1.718204e-13,NaN
4,Report Month,float64,1406,0.529534,-1.0,0.098915,0.027073,NaN,-0.021719
5,ADAS/ADS Hardware Version,str,1206,NaN,NaN,NaN,0.026342,5.766044e-14,NaN
6,Report Type,str,2344,NaN,NaN,NaN,0.025208,4.695040e-27,NaN
7,ADAS/ADS Software Version,str,1207,NaN,NaN,NaN,0.025105,7.323265e-12,NaN
8,Report Year,float64,1406,0.521026,-1.0,0.037607,0.022892,NaN,-0.016166
9,Model,str,2340,NaN,NaN,NaN,0.022709,1.435853e-05,NaN


In [11]:
# Top-20 by each individual metric for side-by-side comparison
for metric, ascending in [
    ('auc', False),
    ('ks', False),
    ('mutual_info', False),
    ('chi2_p', True),  # smaller p = stronger signal
    ('correlation', False),
]:
    sub = ranking.copy()
    if metric == 'correlation':
        sub['abs_corr'] = sub['correlation'].abs()
        sub = sub.sort_values('abs_corr', ascending=False, na_position='last')
        cols = ['feature', 'dtype', 'n_non_null', 'correlation', 'abs_corr']
    else:
        sub = sub.sort_values(metric, ascending=ascending, na_position='last')
        cols = ['feature', 'dtype', 'n_non_null', metric]
    print(f'\n=== top 20 by {metric} ===')
    display(sub[cols].head(20))


=== top 20 by auc ===


,feature,dtype,n_non_null,auc
46,Posted Speed Limit (MPH),float64,1514,0.603654
51,Model Year,float64,2336,0.546216
141,SV Precrash Speed (MPH),float64,2328,0.539302
140,Mileage,float64,1505,0.539089
48,SV Speed >= 15,int64,2344,0.530088
4,Report Month,float64,1406,0.529534
8,Report Year,float64,1406,0.521026
0,Incident Time (24:00),str,2339,NaN
1,Investigating Agency,str,509,NaN
2,Investigating Agency Clean,str,508,NaN



=== top 20 by ks ===


,feature,dtype,n_non_null,ks
46,Posted Speed Limit (MPH),float64,1514,0.180366
51,Model Year,float64,2336,0.102036
140,Mileage,float64,1505,0.100189
4,Report Month,float64,1406,0.098915
141,SV Precrash Speed (MPH),float64,2328,0.073830
48,SV Speed >= 15,int64,2344,0.060176
8,Report Year,float64,1406,0.037607
0,Incident Time (24:00),str,2339,NaN
1,Investigating Agency,str,509,NaN
2,Investigating Agency Clean,str,508,NaN



=== top 20 by mutual_info ===


,feature,dtype,n_non_null,mutual_info
0,Incident Time (24:00),str,2339,0.201118
1,Investigating Agency,str,509,0.054752
2,Investigating Agency Clean,str,508,0.054117
3,ADAS/ADS System Version,str,1514,0.036651
4,Report Month,float64,1406,0.027073
5,ADAS/ADS Hardware Version,str,1206,0.026342
6,Report Type,str,2344,0.025208
7,ADAS/ADS Software Version,str,1207,0.025105
8,Report Year,float64,1406,0.022892
9,Model,str,2340,0.022709



=== top 20 by chi2_p ===


,feature,dtype,n_non_null,chi2_p
1,Investigating Agency,str,509,5.414517e-34
2,Investigating Agency Clean,str,508,9.287501e-34
6,Report Type,str,2344,4.695040e-27
11,Crash With,str,2341,1.429932e-18
25,Law Enforcement Investigating?,str,1532,1.137227e-14
29,CP Contact Area - Front Right,str,525,3.559312e-14
5,ADAS/ADS Hardware Version,str,1206,5.766044e-14
3,ADAS/ADS System Version,str,1514,1.718204e-13
33,Inv. Officer Email - Unknown,str,321,5.957038e-13
7,ADAS/ADS Software Version,str,1207,7.323265e-12



=== top 20 by correlation ===


,feature,dtype,n_non_null,correlation,abs_corr
46,Posted Speed Limit (MPH),float64,1514,0.110982,0.110982
51,Model Year,float64,2336,-0.052016,0.052016
48,SV Speed >= 15,int64,2344,0.048983,0.048983
141,SV Precrash Speed (MPH),float64,2328,0.044107,0.044107
140,Mileage,float64,1505,0.038693,0.038693
4,Report Month,float64,1406,-0.021719,0.021719
8,Report Year,float64,1406,-0.016166,0.016166
0,Incident Time (24:00),str,2339,NaN,NaN
1,Investigating Agency,str,509,NaN,NaN
2,Investigating Agency Clean,str,508,NaN,NaN


## Section 5 -- Modeling prep

Object-dtype features get cast to `category` once so LightGBM can consume them natively. Numerics get median-fill. Rare-bucketing happens *inside* the LogisticRegression `ColumnTransformer` so LightGBM still sees full cardinality while LR sees `__OTHER__` for long-tail levels.

In [12]:
X, y, cat_cols, num_cols = mdl.prepare_modeling_frame(
    treated_df, TARGET_COL, feature_cols,
)
print(f'X shape: {X.shape}; cat_cols: {len(cat_cols)}; num_cols: {len(num_cols)}')
print(f'positive rate (y): {y.mean():.4f}')

# Cardinality of each categorical -- helps spot the high-cardinality cols
# the LR rare-bucketer will collapse.
card = pd.Series({c: int(X[c].nunique()) for c in cat_cols}).sort_values(ascending=False)
print('\ncategorical cardinality (top 15):')
display(card.head(15))

X shape: (2344, 154); cat_cols: 147; num_cols: 7
positive rate (y): 0.0947

categorical cardinality (top 15):


Incident Time (24:00)         1145
ADAS/ADS System Version        134
Investigating Agency           117
Investigating Agency Clean     114
ADAS/ADS Software Version       85
ADAS/ADS Hardware Version       83
Operating Entity                71
Model                           68
Make Model                      67
State or Local Permit           63
Model Clean                     60
Automation Feature Version      59
Incident Date                   58
Report Submission Date          57
Make                            51
dtype: int64

In [13]:
X_train, X_test, y_train, y_test = mdl.stratified_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE,
)
print(f'train: {X_train.shape}, pos rate {y_train.mean():.4f}')
print(f' test: {X_test.shape}, pos rate {y_test.mean():.4f}')

train: (1875, 154), pos rate 0.0949
 test: (469, 154), pos rate 0.0938


## Section 6 -- Three model fits + SHAP

Quick eval, not a deployment claim: 80/20 stratified holdout, `random_state=0`, no cross-validation. SHAP only for the gradient-boosting fit (per plan R5). A second SHAP pass on the pre-incident-only feature set surfaces how much of the signal hangs off co-observed crash outcomes vs upstream cause features.

In [14]:
rf = mdl.fit_lgbm_rf(X_train, y_train, cat_cols, random_state=RANDOM_STATE)
rf_eval = mdl.evaluate_classifier(rf, X_test, y_test, name='lgbm_rf')
print(rf_eval)

rf_imp = mdl.feature_importance_lgbm(rf, list(X.columns))
display(rf_imp.head(20))

{'name': 'lgbm_rf', 'auc': 0.8364973262032086, 'pr_auc': 0.40942751669999, 'n_test': 469, 'n_pos_test': 44}


,feature,gain,split
0,Report Month,74000.326488,270
1,Property Damage?,43685.575794,266
2,Model Year,41784.476894,191
3,CP Contact Area - Front Right,39630.253841,158
4,CP Contact Area - Front,39384.712200,172
5,Report Type,32934.210175,71
6,Inv. Officer Email - Unknown,22995.498390,148
7,SV Precrash Speed (MPH),18996.592716,189
8,Mileage,17316.142948,163
9,SV Were All Passengers Belted?,15960.520126,59


In [15]:
lr = mdl.fit_logistic(X_train, y_train, cat_cols, num_cols, random_state=RANDOM_STATE)
lr_eval = mdl.evaluate_classifier(lr, X_test, y_test, name='logistic_regression')
print(lr_eval)

lr_imp = mdl.feature_importance_logistic(lr)
display(lr_imp.head(20))

{'name': 'logistic_regression', 'auc': 0.8343850267379679, 'pr_auc': 0.31699761549992267, 'n_test': 469, 'n_pos_test': 44}


,feature,coef,abs_coef
0,cat__Crash With_Bus,-1.986834,1.986834
1,cat__SV Were All Passengers Belted?_No Passeng...,-1.835289,1.835289
2,cat__State Clean_GA,1.762949,1.762949
3,cat__Crash With_Non-Motorist: Cyclist,1.580318,1.580318
4,cat__Report Submission Date_SEP-2024,-1.529101,1.529101
5,cat__Were All Passengers Belted?_Subject Vehic...,-1.493935,1.493935
6,cat__Report Type_Monthly,-1.445142,1.445142
7,cat__Property Damage?_Yes,-1.419461,1.419461
8,cat__Report Type_Update,1.377312,1.377312
9,cat__Report Submission Date_NOV-2024,-1.272839,1.272839


In [16]:
gbm = mdl.fit_lgbm_gbm(X_train, y_train, cat_cols, random_state=RANDOM_STATE)
gbm_eval = mdl.evaluate_classifier(gbm, X_test, y_test, name='lgbm_gbm')
print(gbm_eval)

gbm_imp = mdl.feature_importance_lgbm(gbm, list(X.columns))
display(gbm_imp.head(20))

{'name': 'lgbm_gbm', 'auc': 0.8207486631016043, 'pr_auc': 0.3234544610357803, 'n_test': 469, 'n_pos_test': 44}


,feature,gain,split
0,Incident Date,3242.231821,185
1,Report Submission Date,3016.225983,201
2,Mileage,2528.128757,296
3,SV Precrash Speed (MPH),2273.255505,234
4,Report Type,1805.113542,54
5,Property Damage?,1616.581418,106
6,SV Were All Passengers Belted?,1613.131219,41
7,Were All Passengers Belted?,1387.536355,54
8,Report Month,1330.519238,81
9,Model Year,1322.676886,80


In [17]:
(ARTIFACT_ROOT / 'models').mkdir(exist_ok=True)

eval_df = pd.DataFrame([rf_eval, lr_eval, gbm_eval])
eval_df.to_csv(ARTIFACT_ROOT / 'models' / 'eval_summary.csv', index=False)
display(eval_df)

rf_imp.to_csv(ARTIFACT_ROOT / 'models' / 'lgbm_rf_importance.csv', index=False)
lr_imp.to_csv(ARTIFACT_ROOT / 'models' / 'logistic_importance.csv', index=False)
gbm_imp.to_csv(ARTIFACT_ROOT / 'models' / 'lgbm_gbm_importance.csv', index=False)

,name,auc,pr_auc,n_test,n_pos_test
0,lgbm_rf,0.836497,0.409428,469,44
1,logistic_regression,0.834385,0.316998,469,44
2,lgbm_gbm,0.820749,0.323454,469,44


In [18]:
# SHAP on the gradient-boosting fit, using the full feature set.
shap_full = mdl.shap_importance(gbm, X_test, feature_cols=list(X.columns))
shap_full.to_csv(ARTIFACT_ROOT / 'models' / 'shap_importance.csv', index=False)
display(shap_full.head(20))

mdl.shap_summary_plot(
    gbm, X_test,
    out_path=ARTIFACT_ROOT / 'models' / 'shap_summary_bar.png',
    plot_type='bar', max_display=20,
)
mdl.shap_summary_plot(
    gbm, X_test,
    out_path=ARTIFACT_ROOT / 'models' / 'shap_summary_beeswarm.png',
    plot_type='beeswarm', max_display=20,
)

,feature,mean_abs_shap
0,SV Were All Passengers Belted?,0.688578
1,Incident Date,0.656233
2,Report Submission Date,0.606048
3,Were All Passengers Belted?,0.440242
4,CP Pre-Crash Movement,0.394534
5,Investigating Agency Clean,0.390848
6,Report Type,0.367184
7,Crash With,0.273639
8,Property Damage?,0.206253
9,Mileage,0.202967


WindowsPath('artifacts_target_injury/models/shap_summary_beeswarm.png')

In [19]:
# Contrast pass: refit GBM on the pre-incident-only feature set so SHAP
# surfaces upstream signal rather than co-observed crash outcomes.
X_pre, y_pre, cat_cols_pre, num_cols_pre = mdl.prepare_modeling_frame(
    treated_df, TARGET_COL, pre_incident_feature_cols,
)
X_pre_train, X_pre_test, y_pre_train, y_pre_test = mdl.stratified_split(
    X_pre, y_pre, test_size=0.2, random_state=RANDOM_STATE,
)
gbm_pre = mdl.fit_lgbm_gbm(X_pre_train, y_pre_train, cat_cols_pre,
                            random_state=RANDOM_STATE)
gbm_pre_eval = mdl.evaluate_classifier(gbm_pre, X_pre_test, y_pre_test,
                                       name='lgbm_gbm_pre_incident')
print(gbm_pre_eval)

shap_pre = mdl.shap_importance(gbm_pre, X_pre_test, feature_cols=list(X_pre.columns))
shap_pre.to_csv(ARTIFACT_ROOT / 'models' / 'shap_importance_pre_incident.csv', index=False)
display(shap_pre.head(20))

mdl.shap_summary_plot(
    gbm_pre, X_pre_test,
    out_path=ARTIFACT_ROOT / 'models' / 'shap_summary_bar_pre_incident.png',
    plot_type='bar', max_display=20,
)

{'name': 'lgbm_gbm_pre_incident', 'auc': 0.8334224598930481, 'pr_auc': 0.36321217510574266, 'n_test': 469, 'n_pos_test': 44}


,feature,mean_abs_shap
0,SV Were All Passengers Belted?,0.705413
1,Incident Date,0.608571
2,Report Submission Date,0.537427
3,CP Pre-Crash Movement,0.403341
4,Investigating Agency Clean,0.390115
5,Were All Passengers Belted?,0.389112
6,Report Type,0.368186
7,Crash With,0.315380
8,Property Damage?,0.231278
9,Mileage,0.203599


WindowsPath('artifacts_target_injury/models/shap_summary_bar_pre_incident.png')

In [20]:
# Compare SHAP and LightGBM gain rankings side-by-side. They measure
# different things (mean magnitude of marginal contribution vs total
# split-gain) so divergence is informative, not a failure.
compare = (
    shap_full.merge(gbm_imp, on='feature', how='outer')
             .sort_values('mean_abs_shap', ascending=False, na_position='last')
)
display(compare.head(20))

,feature,mean_abs_shap,gain,split
115,SV Were All Passengers Belted?,0.688578,1613.131219,41.0
44,Incident Date,0.656233,3242.231821,185.0
82,Report Submission Date,0.606048,3016.225983,201.0
150,Were All Passengers Belted?,0.440242,1387.536355,54.0
27,CP Pre-Crash Movement,0.394534,1249.430757,85.0
53,Investigating Agency Clean,0.390848,1058.803709,37.0
83,Report Type,0.367184,1805.113542,54.0
30,Crash With,0.273639,1206.207540,123.0
79,Property Damage?,0.206253,1616.581418,106.0
62,Mileage,0.202967,2528.128757,296.0


## Section 7 -- Two-way interactions

Top-K features driven by the full-set SHAP ranking. Pairwise target-rate heatmaps + a depth-3 stub `DecisionTreeClassifier`. The stub tree is for interaction discovery ("which two-feature splits beat which?"), not prediction.

In [21]:
top_k_features = shap_full.head(TOP_K_INTERACTIONS)['feature'].tolist()
print(f'top-{TOP_K_INTERACTIONS} features by SHAP:')
for f in top_k_features:
    print(f'  - {f}')

top-8 features by SHAP:
  - SV Were All Passengers Belted?
  - Incident Date
  - Report Submission Date
  - Were All Passengers Belted?
  - CP Pre-Crash Movement
  - Investigating Agency Clean
  - Report Type
  - Crash With


In [22]:
heatmap_paths = ix.pairwise_heatmaps_to_png(
    treated_df, top_k_features, TARGET_COL,
    out_dir=ARTIFACT_ROOT / 'interactions' / 'pairwise_heatmaps',
)
print(f'wrote {len(heatmap_paths)} heatmap PNGs')

wrote 28 heatmap PNGs


In [23]:
tree, tree_features = ix.fit_stub_tree(
    treated_df, top_k_features, TARGET_COL, max_depth=3,
)
tree_text = ix.stub_tree_text(tree, tree_features)
print(tree_text)

(ARTIFACT_ROOT / 'interactions').mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / 'interactions' / 'stub_tree.txt').write_text(
    tree_text, encoding='utf-8'
)
ix.stub_tree_png(
    tree, tree_features,
    out_path=ARTIFACT_ROOT / 'interactions' / 'stub_tree.png',
)

|--- Investigating Agency Clean <= 2.50
|   |--- SV Were All Passengers Belted? <= 0.50
|   |   |--- Report Type <= 1.50
|   |   |   |--- class: 1
|   |   |--- Report Type >  1.50
|   |   |   |--- class: 0
|   |--- SV Were All Passengers Belted? >  0.50
|   |   |--- CP Pre-Crash Movement <= 17.50
|   |   |   |--- class: 0
|   |   |--- CP Pre-Crash Movement >  17.50
|   |   |   |--- class: 1
|--- Investigating Agency Clean >  2.50
|   |--- Report Type <= 2.50
|   |   |--- SV Were All Passengers Belted? <= 0.50
|   |   |   |--- class: 1
|   |   |--- SV Were All Passengers Belted? >  0.50
|   |   |   |--- class: 1
|   |--- Report Type >  2.50
|   |   |--- Report Type <= 4.00
|   |   |   |--- class: 0
|   |   |--- Report Type >  4.00
|   |   |   |--- class: 1



WindowsPath('artifacts_target_injury/interactions/stub_tree.png')

## Notes

* Top SHAP features (full set) are dominated by co-observed crash outcomes (towed / airbag / speed). The pre-incident-only contrast SHAP pass shows what survives once those are removed -- a useful sanity check on whether the gradient-boosting model is learning anything beyond "a bad crash had injuries."
* The univariate ranking and the SHAP ranking measure different things; spot-checks where they disagree are usually high-cardinality categoricals (chi-square loves them, SHAP gain doesn't always).
* Single 80/20 stratified holdout, no cross-validation. The 222 positives are split ~178 / 44 -- holdout AUC noise is real. Treat this as a triage layer, not a model claim.
* Heatmap pairs below `min_cell_count = ceil(3 / positive_rate) ~ 32` are skipped (logged inline) so the artifact tree stays signal-heavy.

**Next target to mirror this on**: `SV Speed >= 15` -- listed in `eda_to_do.md` line 58, deferred to a follow-up plan. All three util files take a target column as input, so the follow-up should be one notebook + one new artifact subtree, no new utils.

In [ ]:
'''
Human rough notes

notes on target EDA
by reporting entity, waymo seems to have smaller positive rate, Zoox and Cruise higher. TEsla slightly higher
monthly reports have the lowest rate. guess other oens are update dmore frequently due to severity
	clean operating entity
		waymo around average, zoox and cruise higher, tesla a bit higher

month relative stable, maybe more in nov or dec. higher rate in 2022 and 2023, maybe from cruise?

intersection highest rate, a bit above street.
work zone a bit higher but small, same with traffic circle
higher speeds generally except not 70 + for some reason
	would want to bucke this more

huh not much there with weather

crash with seems to have as expected but small sample
	ie hire rate with non motorist, motorcycle, cyclist

pre CP crash movement has some stuff that might be interesting
	for SV seems to be making turns mainly and some others
hardly any CP airbags or towing, but htis data is largely old only
	new one seems to have higher accident rates with the airbags going off which maeks esne
	twoed as well

huh SV speed doesn't seem to matter. maybe slightly but generally not moving that fast
	so fast CP hitting slow SV

law enforcement investigating hire but still only 20%

not much tghere with the roadway condition




Univariate
not much tehre. mabye posted speed limit
maybe incicednt time 

LGBM gets something decent
	can see some stuff ther but not clear why soem of the features are important

LR
	some of the crash with makes sense and the not bellted un clear why some of the others re important

important features
	passengers belted (maeks sense), pre crash movement and crash with make sense
	some after the facto fcators matter more


backlog:
crash with other fixed object and and other see narrative might be worth reasing
certain car or cars accounting for more rates
contact areas and the movemetns from both
	maybe for the website
for incident time anything in there?
this with pre contact moving only
have thea gent run this analysi and see waht is important

'''